# 00 — Fetch and store ENFUSER PM₂.₅ for two Locomizer timestamps

## Purpose

This companion notebook tests and stores FMI ENFUSER PM₂.₅ data for the two proof-of-concept timestamps used in Stage 05:

- **Time A:** 2023-06-17 12:00 Helsinki time = **2023-06-17 09:00 UTC**
- **Time B:** 2023-06-18 12:00 Helsinki time = **2023-06-18 09:00 UTC**

The notebook is intentionally separate from the analysis workflow.

It:

1. queries the FMI WFS ENFUSER Helsinki-metropolitan stored query;
2. saves the **raw XML response** for reproducibility;
3. checks whether the requested historical timestamp is actually available;
4. extracts any download / coverage URLs present in the response;
5. saves metadata needed by Stage 05.

## Important

The FMI stored query used here is:

```text
fmi::forecast::enfuser::airquality::helsinki-metropolitan::grid
```

with parameter:

```text
PM25Concentration
```

Historical availability for June 2023 must be tested rather than assumed. If FMI's live WFS no longer exposes those historical grids, the raw response saved by this notebook will make that clear and we can then switch to an archive source.


## 1. Imports and storage paths

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import re
import json

import pandas as pd
import requests
from lxml import etree

# Project storage
BASE_DIR = Path(
    "/scratch/project_2010417/paper3/locomizer/derived/enfuser"
)

RAW_DIR = BASE_DIR / "raw"
META_DIR = BASE_DIR / "metadata"
DATA_DIR = BASE_DIR / "data"

for folder in [RAW_DIR, META_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Storage root:", BASE_DIR)


Storage root: /scratch/project_2010417/paper3/locomizer/derived/enfuser


## 2. Configure the two timestamps

In [2]:
# Helsinki is UTC+3 in June.
# 12:00 Helsinki local time = 09:00 UTC.

TIMES = {
    "time_A": {
        "local_time": "2023-06-17 12:00",
        "utc_time": "2023-06-17T09:00:00Z",
    },
    "time_B": {
        "local_time": "2023-06-18 12:00",
        "utc_time": "2023-06-18T09:00:00Z",
    },
}

for key, value in TIMES.items():
    print(key, value)


time_A {'local_time': '2023-06-17 12:00', 'utc_time': '2023-06-17T09:00:00Z'}
time_B {'local_time': '2023-06-18 12:00', 'utc_time': '2023-06-18T09:00:00Z'}


## 3. FMI ENFUSER WFS configuration

In [3]:
FMI_WFS_URL = "https://opendata.fmi.fi/wfs"

STORED_QUERY = (
    "fmi::forecast::enfuser::airquality::helsinki-metropolitan::grid"
)

PARAMETER = "PM25Concentration"

# Optional:
# If you later want to restrict the request spatially, set a WGS84 bbox:
#
# [min_lon, min_lat, max_lon, max_lat]
#
# Example Helsinki-region-like extent:
# BBOX = [24.4, 60.05, 25.3, 60.45]
#
# For the first diagnostic request, leave it as None.
BBOX = None


## 4. Query helper

The raw response is always saved, even if FMI returns an error or an empty historical result.


In [4]:
def fetch_enfuser_response(label, timestamp_utc):
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "getFeature",
        "storedquery_id": STORED_QUERY,
        "starttime": timestamp_utc,
        "endtime": timestamp_utc,
        "parameters": PARAMETER,
    }

    if BBOX is not None:
        params["bbox"] = ",".join(map(str, BBOX))

    response = requests.get(
        FMI_WFS_URL,
        params=params,
        timeout=120,
    )

    raw_path = RAW_DIR / f"{label}_pm25_wfs.xml"
    raw_path.write_bytes(response.content)

    return response, raw_path


## 5. Run the two FMI requests

In [5]:
request_rows = []
responses = {}

for label, info in TIMES.items():
    print()
    print("=" * 70)
    print(label, info["local_time"], "/", info["utc_time"])

    response, raw_path = fetch_enfuser_response(
        label,
        info["utc_time"],
    )

    responses[label] = response

    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("content-type"))
    print("Bytes:", len(response.content))
    print("Saved raw response:", raw_path)

    preview = response.text[:700].replace("\n", " ")
    print("Preview:")
    print(preview)

    request_rows.append({
        "label": label,
        "local_time": info["local_time"],
        "utc_time": info["utc_time"],
        "http_status": response.status_code,
        "content_type": response.headers.get("content-type"),
        "response_bytes": len(response.content),
        "raw_response_path": str(raw_path),
        "request_url": response.url,
    })

request_log = pd.DataFrame(request_rows)
display(request_log)



time_A 2023-06-17 12:00 / 2023-06-17T09:00:00Z
HTTP status: 200
Content type: text/xml; charset=UTF-8
Bytes: 1743
Saved raw response: /scratch/project_2010417/paper3/locomizer/derived/enfuser/raw/time_A_pm25_wfs.xml
Preview:
<?xml version="1.0" encoding="UTF-8"?> <wfs:FeatureCollection     timeStamp="2026-10-05T20:18:18Z"     numberMatched="0"     numberReturned="0"         xmlns:wfs="http://www.opengis.net/wfs/2.0"     xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"     xmlns:xlink="http://www.w3.org/1999/xlink"     xmlns:om="http://www.opengis.net/om/2.0"     xmlns:omso="http://inspire.ec.europa.eu/schemas/omso/3.0"     xmlns:gml="http://www.opengis.net/gml/3.2"     xmlns:gmd="http://www.isotc211.org/2005/gmd"     xmlns:swe="http://www.opengis.net/swe/2.0"     xmlns:gmlcov="http://www.opengis.net/gmlcov/1.0"     xmlns:inspire="urn:x-inspire:specification:gmlas:BaseTypes:3.2"     xmlns:sam="http://www.ope

time_B 2023-06-18 12:00 / 2023-06-18T09:00:00Z
HTTP status: 200
Content 

,label,local_time,utc_time,http_status,content_type,response_bytes,raw_response_path,request_url
0,time_A,2023-06-17 12:00,2023-06-17T09:00:00Z,200,text/xml; charset=UTF-8,1743,/scratch/project_2010417/paper3/locomizer/deri...,https://opendata.fmi.fi/wfs?service=WFS&versio...
1,time_B,2023-06-18 12:00,2023-06-18T09:00:00Z,200,text/xml; charset=UTF-8,1743,/scratch/project_2010417/paper3/locomizer/deri...,https://opendata.fmi.fi/wfs?service=WFS&versio...


## 6. Inspect XML for exception messages and URLs

This cell searches the WFS response for:

- OGC/FMI exception text;
- URLs embedded in the XML;
- references that may point to downloadable grid/coverage data.

It does not assume a particular response structure.


In [6]:
URL_PATTERN = re.compile(
    r'https?://[^\s\"\'<>]+'
)

inspection_rows = []

for label, response in responses.items():
    text = response.text

    urls = sorted(set(URL_PATTERN.findall(text)))

    exception_texts = []

    try:
        root = etree.fromstring(response.content)

        # Capture text from exception-related XML nodes if present.
        for element in root.iter():
            tag = element.tag
            if isinstance(tag, str) and (
                "Exception" in tag
                or "exception" in tag
            ):
                value = " ".join(element.itertext()).strip()
                if value:
                    exception_texts.append(value)

    except Exception as exc:
        exception_texts.append(
            f"XML parsing failed: {exc}"
        )

    inspection_rows.append({
        "label": label,
        "n_urls_found": len(urls),
        "urls": urls,
        "exception_text": " | ".join(exception_texts),
    })

inspection = pd.DataFrame(inspection_rows)
display(inspection)


,label,n_urls_found,urls,exception_text
0,time_A,20,"[http://inspire.ec.europa.eu/schemas/omso/3.0,...",
1,time_B,20,"[http://inspire.ec.europa.eu/schemas/omso/3.0,...",


## 7. Save request and inspection metadata

In [7]:
request_log_path = META_DIR / "enfuser_two_time_request_log.csv"
inspection_path = META_DIR / "enfuser_two_time_response_inspection.json"

request_log.to_csv(
    request_log_path,
    index=False,
)

inspection.to_json(
    inspection_path,
    orient="records",
    indent=2,
)

print("Saved:", request_log_path)
print("Saved:", inspection_path)


Saved: /scratch/project_2010417/paper3/locomizer/derived/enfuser/metadata/enfuser_two_time_request_log.csv
Saved: /scratch/project_2010417/paper3/locomizer/derived/enfuser/metadata/enfuser_two_time_response_inspection.json


## 8. Detect whether usable historical ENFUSER content appears to be present

This is a conservative diagnostic.

A successful HTTP `200` alone does **not** prove that the requested 2023 grid exists. We also look for obvious exception text or an empty response.


In [8]:
def classify_response(response):
    text_lower = response.text.lower()

    if response.status_code != 200:
        return "http_error"

    if (
        "exceptionreport" in text_lower
        or "exceptiontext" in text_lower
        or "serviceexception" in text_lower
    ):
        return "wfs_exception"

    if len(response.content) < 500:
        return "very_small_response"

    return "inspect_manually"

availability_rows = []

for label, response in responses.items():
    availability_rows.append({
        "label": label,
        "status": classify_response(response),
        "bytes": len(response.content),
    })

availability = pd.DataFrame(availability_rows)
display(availability)


,label,status,bytes
0,time_A,inspect_manually,1743
1,time_B,inspect_manually,1743


## 9. Optional: download referenced files automatically

If the response contains direct HTTP(S) links to data products, this block can download likely raster/grid files.

It intentionally skips generic FMI website links and only attempts URLs whose suffix or query suggests a downloadable geospatial product.

If no suitable URL is found, this block will simply report that.


In [9]:
DOWNLOAD_HINTS = (
    ".tif",
    ".tiff",
    ".nc",
    ".grib",
    ".grb",
    ".zip",
    "coverage",
    "download",
)

download_records = []

for row in inspection.to_dict("records"):
    label = row["label"]
    urls = row["urls"]

    candidates = [
        url for url in urls
        if any(hint in url.lower() for hint in DOWNLOAD_HINTS)
    ]

    if not candidates:
        print(label, ": no obvious downloadable raster/grid URL found.")
        continue

    for i, url in enumerate(candidates, start=1):
        print(label, "candidate:", url)

        try:
            r = requests.get(url, timeout=180)

            suffix = Path(urlparse(url).path).suffix
            if not suffix:
                suffix = ".bin"

            destination = DATA_DIR / f"{label}_candidate_{i}{suffix}"
            destination.write_bytes(r.content)

            download_records.append({
                "label": label,
                "url": url,
                "http_status": r.status_code,
                "bytes": len(r.content),
                "saved_path": str(destination),
            })

            print("  saved:", destination)

        except Exception as exc:
            download_records.append({
                "label": label,
                "url": url,
                "http_status": None,
                "bytes": None,
                "saved_path": None,
                "error": str(exc),
            })

downloads = pd.DataFrame(download_records)

if not downloads.empty:
    display(downloads)

    downloads.to_csv(
        META_DIR / "enfuser_candidate_downloads.csv",
        index=False,
    )


time_A : no obvious downloadable raster/grid URL found.
time_B : no obvious downloadable raster/grid URL found.


## 10. Create a small manifest for Stage 05

This manifest is the hand-off point to the Locomizer analysis notebook.

If actual PM₂.₅ files have been downloaded, update `pm25_path` below to the correct file for each timestamp.

Stage 05 can then read this manifest rather than hard-coding file paths.


In [10]:
manifest = pd.DataFrame([
    {
        "label": "time_A",
        "local_time": TIMES["time_A"]["local_time"],
        "utc_time": TIMES["time_A"]["utc_time"],
        "parameter": PARAMETER,
        "source": "FMI ENFUSER",
        "pm25_path": None,
    },
    {
        "label": "time_B",
        "local_time": TIMES["time_B"]["local_time"],
        "utc_time": TIMES["time_B"]["utc_time"],
        "parameter": PARAMETER,
        "source": "FMI ENFUSER",
        "pm25_path": None,
    },
])

manifest_path = META_DIR / "enfuser_two_time_manifest.csv"
manifest.to_csv(manifest_path, index=False)

display(manifest)
print("Saved:", manifest_path)


,label,local_time,utc_time,parameter,source,pm25_path
0,time_A,2023-06-17 12:00,2023-06-17T09:00:00Z,PM25Concentration,FMI ENFUSER,None
1,time_B,2023-06-18 12:00,2023-06-18T09:00:00Z,PM25Concentration,FMI ENFUSER,None


Saved: /scratch/project_2010417/paper3/locomizer/derived/enfuser/metadata/enfuser_two_time_manifest.csv


## 11. What to do after running this notebook

### If the historical grids are available

You should end up with two reusable files under:

```text
/scratch/project_2010417/paper3/locomizer/derived/enfuser/data/
```

Then Stage 05 can point:

```python
PM25_A_PATH = "..."
PM25_B_PATH = "..."
```

to those files.

### If FMI returns an exception / no 2023 data

Keep the saved XML and metadata. They document that the live WFS route was tested and did not provide the historical grid.

The next step would then be to locate an FMI/ENFUSER archive or another historical source rather than substituting the single-hour CAFEIN sample raster.


In [11]:
for label, response in responses.items():

    print("\n" + "=" * 80)
    print(label)
    print("HTTP:", response.status_code)
    print("bytes:", len(response.content))

    text = response.text

    # Look for the most informative parts
    keywords = [
        "Exception",
        "exception",
        "2023",
        "PM25",
        "Coverage",
        "coverage",
        "fileReference",
        "http",
    ]

    for keyword in keywords:
        if keyword.lower() in text.lower():
            print(f"\nFOUND: {keyword}")

    print("\nFIRST 3000 CHARACTERS:")
    print(text[:3000])


time_A
HTTP: 200
bytes: 1743

FOUND: http

FIRST 3000 CHARACTERS:
<?xml version="1.0" encoding="UTF-8"?>
<wfs:FeatureCollection
    timeStamp="2026-10-05T20:18:18Z"
    numberMatched="0"
    numberReturned="0"
        xmlns:wfs="http://www.opengis.net/wfs/2.0"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xlink="http://www.w3.org/1999/xlink"
    xmlns:om="http://www.opengis.net/om/2.0"
    xmlns:omso="http://inspire.ec.europa.eu/schemas/omso/3.0"
    xmlns:gml="http://www.opengis.net/gml/3.2"
    xmlns:gmd="http://www.isotc211.org/2005/gmd"
    xmlns:swe="http://www.opengis.net/swe/2.0"
    xmlns:gmlcov="http://www.opengis.net/gmlcov/1.0"
    xmlns:inspire="urn:x-inspire:specification:gmlas:BaseTypes:3.2"
    xmlns:sam="http://www.opengis.net/sampling/2.0"
    xmlns:sams="http://www.opengis.net/samplingSpatial/2.0" 
    xmlns:target="http://xml.fmi.fi/namespace/om/atmosphericfeatures/1.1"
    xsi:schemaLocation="
    http://www.opengis.net/wfs/2.0 http://schemas.